### The ReAct Pattern

* ReAct = Reasoning + Acting. It's a way of making the LLM think out loud in a structured text format — every step is written as a Thought, an Action, an Action Input, and an Observation — so you can read exactly how the agent reasoned its way to the answer.

* Unlike tool-calling agents (which output silent JSON), ReAct agents show their work, making them auditable and debuggable. This pattern is used in production wherever the reasoning trail matters — banking, healthcare, legal, and any AI system that must explain its decisions.

In [1]:
import os
import logging
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import AgentExecutor, create_react_agent

In [2]:
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
logging.getLogger('httpx').setLevel(logging.WARNING)

In [3]:
load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

Two Simple Tools

In [4]:
@tool
def search_crop_disease(query: str) -> str:
    '''
    Search the crop disease knowledge base for disease information and treatment.
    '''
    
    knowledge = {
        'cassava mosaic': 'Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, plant resistant varieties, remove infected plants.',
        'maize smut':     'Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.',
        'rice blast':     'Rice Blast: fungal, causes diamond-shaped lesions on leaves. Control: use resistant varieties, balance nitrogen, apply fungicides when needed.',
    }
    
    q = query.lower()
    for key, value in knowledge.items():
        if key in q:
            return value
    return f'No knowledge found for: {query}'
    
@tool
def get_weather(city: str) -> str:
    '''
    Return the current weather for a city.
    '''
    
    fake = {
        'lagos': 'Sunny, 32°C, humidity 78%',
        'abuja': 'Cloudy, 28°C, humidity 65%',
        'kano':  'Hot and dry, 36°C, humidity 30%',
    }
    
    return fake.get(city.lower(), f'No weather data for {city}')

print('Two tools ready')
print(f'   - {search_crop_disease.name}')
print(f'   - {get_weather.name}')

Two tools ready
   - search_crop_disease
   - get_weather


The ReAct Prompt Template

In [5]:
react_template = '''
Answer the following question as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought: {agent_scratchpad}
'''

prompt = PromptTemplate.from_template(react_template)

print('ReAct prompt template ready')


ReAct prompt template ready


Build the ReAct Agent and Executor

In [6]:
tools = [search_crop_disease, get_weather]

# Builds the agent
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5,     # stop after 5 loops
    max_execution_time=30,  # stop after 30 seconds
)

print('ReAct agent ready')

ReAct agent ready


ReAct Test

In [7]:
agent_executor.invoke({'input': 'How do I treat maize smut?'})



> Entering new AgentExecutor chain...
I need to find information on maize smut, including its treatment options. I'll use the search_crop_disease tool to look up this disease.

Action: search_crop_disease  
Action Input: "maize smut treatment"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information on maize smut and its treatment options. The disease is fungal and causes grey or black galls on ears and stalks. The recommended control measures include planting resistant hybrids, rotating crops, and removing galls before they burst.

Final Answer: To treat maize smut, plant resistant hybrids, rotate crops, and remove galls before they burst.

> Finished chain.


{'input': 'How do I treat maize smut?',
 'output': 'To treat maize smut, plant resistant hybrids, rotate crops, and remove galls before they burst.'}

In [8]:
agent_executor.invoke({'input': 'How do I treat maize smut in Yoruba?'})



> Entering new AgentExecutor chain...
I need to find information on treating maize smut, specifically in the Yoruba language. I'll search the crop disease knowledge base for relevant information.

Action: search_crop_disease  
Action Input: "maize smut treatment in Yoruba"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information on maize smut treatment, but it is in English. I need to summarize the treatment methods in a way that can be understood in Yoruba.

Action: search_crop_disease  
Action Input: "maize smut treatment"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have the treatment methods for maize smut, but I need to translate and summarize them into Yoruba for better understanding.

Action: search_crop_disease  
Action Input: "maize smut treatment in Yoruba"  

{'input': 'How do I treat maize smut in Yoruba?',
 'output': 'Lati tọju aisan maize smut, o le yan irugbin ti o ni aabo si aisan yii, yi awọn irugbin pada ni gbogbo ọdun, ati yọ awọn galls kuro ki wọn to fọ.'}

In [9]:
agent_executor.invoke({'input': 'How do I treat maize smut in both Yoruba, Hausa, Igbo, Pidgin, French and Arbic?'})



> Entering new AgentExecutor chain...
I need to search for information on maize smut treatment in the crop disease knowledge base. 

Action: search_crop_disease  
Action Input: "maize smut treatment"  Maize Smut: fungal, causes grey or black galls on ears and stalks. Control: plant resistant hybrids, rotate crops, remove galls before they burst.I have found information on how to treat maize smut. Now, I need to translate this information into Yoruba, Hausa, Igbo, Pidgin, French, and Arabic.

1. **Yoruba**: "Igbẹ́kẹ̀lé: Kó àwọn irugbin tí ó ní ìdáhùn, yípadà irugbin, yọ àwọn galls kí wọn tó fọ."
2. **Hausa**: "Kulawa: Shuka nau'in da ke da juri, canza shuka, cire galls kafin su fashe."
3. **Igbo**: "Nlekọta: Tụọ ụdị nwere nchebe, gbanwee mkpụrụ, wepụ galls tupu ha agbapụ."
4. **Pidgin**: "Treatment: Plant resistant type, change crops, remove galls before dem burst."
5. **French**: "Contrôle: Plantez des hybrides résistants, faites une rotation des cultures, enlevez les galls avant qu'

{'input': 'How do I treat maize smut in both Yoruba, Hausa, Igbo, Pidgin, French and Arbic?',
 'output': 'The treatment for maize smut is as follows:\n- **Yoruba**: "Igbẹ́kẹ̀lé: Kó àwọn irugbin tí ó ní ìdáhùn, yípadà irugbin, yọ àwọn galls kí wọn tó fọ."\n- **Hausa**: "Kulawa: Shuka nau\'in da ke da juri, canza shuka, cire galls kafin su fashe."\n- **Igbo**: "Nlekọta: Tụọ ụdị nwere nchebe, gbanwee mkpụrụ, wepụ galls tupu ha agbapụ."\n- **Pidgin**: "Treatment: Plant resistant type, change crops, remove galls before dem burst."\n- **French**: "Contrôle: Plantez des hybrides résistants, faites une rotation des cultures, enlevez les galls avant qu\'ils n\'éclatent."\n- **Arabic**: "العلاج: زرع أصناف مقاومة، تغيير المحاصيل، إزالة الكتل قبل أن تنفجر."'}

Multi-Tool Test

In [10]:
agent_executor.invoke({
    'input': 'What is the weather in Lagos, and how do I treat cassava mosaic disease?'
})



> Entering new AgentExecutor chain...
I need to gather information about the current weather in Lagos and also look up treatment options for cassava mosaic disease. 

Action: get_weather  
Action Input: "Lagos"  Sunny, 32°C, humidity 78%I have the current weather information for Lagos. Now, I need to find out how to treat cassava mosaic disease.

Action: search_crop_disease  
Action Input: "cassava mosaic disease"  Cassava Mosaic Disease: viral, causes yellowing and mottling of leaves. Control: use disease-free cuttings, plant resistant varieties, remove infected plants.I now have both the weather information for Lagos and the treatment options for cassava mosaic disease. 

Final Answer: The current weather in Lagos is sunny with a temperature of 32°C and humidity at 78%. To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants.

> Finished chain.


{'input': 'What is the weather in Lagos, and how do I treat cassava mosaic disease?',
 'output': 'The current weather in Lagos is sunny with a temperature of 32°C and humidity at 78%. To treat cassava mosaic disease, use disease-free cuttings, plant resistant varieties, and remove infected plants.'}